# DrugRec Data Processing (Official)

This notebook contains the minimal, reproducible preprocessing pipeline for the DrugRec task.
Final output: `diag_adj.csv`.


In [ ]:
import pandas as pd
import numpy as np

# Input files
med_file = './input/prescriptions.csv'
diag_file = './input/preproc_diag1.csv.gz'
procedure_file = './input/procedures_icd.csv'
RXCUI2atc4_file = './input/RXCUI2atc4.csv'
rxnorm2RXCUI_file = './input/rxnorm2RXCUI.txt'
drugbankinfo = './input/drugbank_drugs_info.csv'

# Output files
processed_med_file = './output/processed_med2.csv'
combined_data_file = './output/data2.csv'
diag_adj_file = './output/diag_adj.csv'

In [ ]:
def med_process(med_file):
    med_pd = pd.read_csv(med_file, dtype={'ndc': 'category'})
    med_pd.drop(
        columns=[
            'poe_id', 'poe_seq', 'order_provider_id', 'drug_type',
            'formulary_drug_cd', 'prod_strength', 'dose_val_rx',
            'dose_unit_rx', 'form_val_disp', 'form_unit_disp', 'gsn',
            'route', 'stoptime'
        ],
        axis=1,
        inplace=True,
    )
    med_pd.drop(index=med_pd[med_pd['ndc'] == '0'].index, axis=0, inplace=True)
    med_pd.fillna(method='pad', inplace=True)
    med_pd.dropna(inplace=True)
    med_pd.drop_duplicates(inplace=True)
    med_pd['starttime'] = pd.to_datetime(med_pd['starttime'], format='%Y-%m-%d %H:%M:%S')
    med_pd.sort_values(by=['subject_id', 'hadm_id', 'starttime'], inplace=True)
    med_pd = med_pd.reset_index(drop=True)
    med_pd = med_pd.drop_duplicates().reset_index(drop=True)
    return med_pd


def process_visit_lg2(med_pd):
    visits = med_pd[['subject_id', 'hadm_id']].groupby(by='subject_id')['hadm_id'].unique().reset_index()
    visits['hadm_id_len'] = visits['hadm_id'].map(len)
    return visits[visits['hadm_id_len'] > 1]


def codeMapping2atc4(med_pd):
    with open(rxnorm2RXCUI_file, 'r') as f:
        rxnorm2RXCUI = eval(f.read())
    med_pd['RXCUI'] = med_pd['ndc'].map(rxnorm2RXCUI)
    med_pd.dropna(inplace=True)

    rxnorm2atc4 = pd.read_csv(RXCUI2atc4_file)
    rxnorm2atc4 = rxnorm2atc4.drop(columns=['YEAR', 'MONTH', 'NDC'])
    rxnorm2atc4.drop_duplicates(subset=['RXCUI'], inplace=True)

    med_pd.drop(index=med_pd[med_pd['RXCUI'].isin([''])].index, axis=0, inplace=True)
    med_pd['RXCUI'] = med_pd['RXCUI'].astype('int64')
    med_pd = med_pd.reset_index(drop=True)

    med_pd = med_pd.merge(rxnorm2atc4, on=['RXCUI'])
    med_pd.drop(columns=['ndc', 'RXCUI'], inplace=True)
    med_pd['ATC4'] = med_pd['ATC4'].map(lambda x: x[:4])
    med_pd = med_pd.rename(columns={'ATC4': 'ATC3'})
    med_pd = med_pd.drop_duplicates().reset_index(drop=True)
    return med_pd


def filter_300_most_med(med_pd):
    med_count = (
        med_pd.groupby(by=['ATC3']).size().reset_index()
        .rename(columns={0: 'count'})
        .sort_values(by=['count'], ascending=False)
        .reset_index(drop=True)
    )
    med_pd = med_pd[med_pd['ATC3'].isin(med_count.loc[:299, 'ATC3'])]
    return med_pd.reset_index(drop=True)


def ATC3toDrug(med_pd):
    atc3toDrugDict = {}
    for atc3, drugname in med_pd[['ATC3', 'drug']].values:
        if atc3 in atc3toDrugDict:
            atc3toDrugDict[atc3].add(drugname)
        else:
            atc3toDrugDict[atc3] = set(drugname)
    return atc3toDrugDict


def atc3_filter_by_drugbank(med_pd, drugbankinfo):
    atc3_to_drug = ATC3toDrug(med_pd)
    druginfo = pd.read_csv(drugbankinfo)
    drug_with_smiles = set(druginfo.loc[druginfo['moldb_smiles'].notna(), 'name'])
    valid_atc3 = [k for k, v in atc3_to_drug.items() if len(set(v) & drug_with_smiles) > 0]
    return med_pd[med_pd['ATC3'].isin(valid_atc3)].reset_index(drop=True)

In [ ]:
def diag_process(diag_pd):
    diag_pd.dropna(inplace=True)
    diag_pd.drop_duplicates(inplace=True)
    diag_pd.sort_values(by=['subject_id', 'hadm_id'], inplace=True)
    diag_pd = diag_pd.reset_index(drop=True)

    diag_count = (
        diag_pd.groupby(by=['new_icd_code']).size().reset_index()
        .rename(columns={0: 'count'})
        .sort_values(by=['count'], ascending=False)
        .reset_index(drop=True)
    )
    diag_pd = diag_pd[diag_pd['new_icd_code'].isin(diag_count.loc[:1999, 'new_icd_code'])]
    return diag_pd.reset_index(drop=True)


def procedure_process(procedure_file):
    pro_pd = pd.read_csv(procedure_file, dtype={'ICD_CODE': 'category'})
    pro_pd.drop_duplicates(inplace=True)
    pro_pd.sort_values(by=['SUBJECT_ID', 'HADM_ID', 'SEQ_NUM'], inplace=True)
    pro_pd.drop_duplicates(inplace=True)
    pro_pd.reset_index(drop=True, inplace=True)
    return pro_pd


def filter_1000_most_pro(pro_pd):
    pro_count = (
        pro_pd.groupby(by=['ICD_CODE']).size().reset_index()
        .rename(columns={0: 'count'})
        .sort_values(by=['count'], ascending=False)
        .reset_index(drop=True)
    )
    pro_pd = pro_pd[pro_pd['ICD_CODE'].isin(pro_count.loc[:999, 'ICD_CODE'])]
    return pro_pd.reset_index(drop=True)


def combine_process(med_pd, diag_pd, pro_pd):
    med_pd = med_pd.rename(columns={'subject_id': 'SUBJECT_ID', 'hadm_id': 'HADM_ID'})
    diag_pd = diag_pd.rename(columns={'subject_id': 'SUBJECT_ID', 'hadm_id': 'HADM_ID'})

    med_pd_key = med_pd[['SUBJECT_ID', 'HADM_ID']].drop_duplicates()
    diag_pd_key = diag_pd[['SUBJECT_ID', 'HADM_ID']].drop_duplicates()
    pro_pd_key = pro_pd[['SUBJECT_ID', 'HADM_ID']].drop_duplicates()

    combined_key = med_pd_key.merge(diag_pd_key, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
    combined_key = combined_key.merge(pro_pd_key, on=['SUBJECT_ID', 'HADM_ID'], how='inner')

    diag_pd = diag_pd.merge(combined_key, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
    med_pd = med_pd.merge(combined_key, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
    pro_pd = pro_pd.merge(combined_key, on=['SUBJECT_ID', 'HADM_ID'], how='inner')

    diag_pd = diag_pd.groupby(by=['SUBJECT_ID', 'HADM_ID'])['new_icd_code'].unique().reset_index()
    med_pd = med_pd.groupby(by=['SUBJECT_ID', 'HADM_ID'])['ATC3'].unique().reset_index()
    pro_pd = (
        pro_pd.groupby(by=['SUBJECT_ID', 'HADM_ID'])['ICD_CODE'].unique().reset_index()
        .rename(columns={'ICD_CODE': 'PRO_CODE'})
    )

    med_pd['ATC3'] = med_pd['ATC3'].map(list)
    pro_pd['PRO_CODE'] = pro_pd['PRO_CODE'].map(list)

    data = diag_pd.merge(med_pd, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
    data = data.merge(pro_pd, on=['SUBJECT_ID', 'HADM_ID'], how='inner')
    return data


def build_diag_adj(diag_pd):
    diag_pd = diag_pd[['subject_id', 'new_icd_code']].drop_duplicates().copy()
    patient_ids = sorted(diag_pd['subject_id'].unique())
    diag_codes = sorted(diag_pd['new_icd_code'].unique())

    patient2idx = {pid: i for i, pid in enumerate(patient_ids)}
    code2idx = {code: j for j, code in enumerate(diag_codes)}

    diag_adj = np.zeros((len(patient_ids), len(diag_codes)), dtype=np.int8)
    for _, row in diag_pd.iterrows():
        diag_adj[patient2idx[row['subject_id']], code2idx[row['new_icd_code']]] = 1

    diag_adj_df = pd.DataFrame(diag_adj, index=patient_ids, columns=diag_codes)
    diag_adj_df.index.name = 'subject_id'
    return diag_adj_df

In [ ]:
# Core preprocessing pipeline

diag_raw = pd.read_csv(diag_file, compression='gzip', header=0)

med_pd = med_process(med_file)
med_pd_lg2 = process_visit_lg2(med_pd).reset_index(drop=True)
med_pd = med_pd.merge(med_pd_lg2[['subject_id']], on='subject_id', how='inner').reset_index(drop=True)
med_pd = codeMapping2atc4(med_pd)
med_pd = filter_300_most_med(med_pd)
med_pd = atc3_filter_by_drugbank(med_pd, drugbankinfo)
med_pd.to_csv(processed_med_file, index=False)
print('complete medication processing')

diag_pd = diag_process(diag_raw)
print('complete diagnosis processing')

pro_pd = procedure_process(procedure_file)
pro_pd = filter_1000_most_pro(pro_pd)
print('complete procedure processing')

data = combine_process(med_pd, diag_pd, pro_pd)
data.to_csv(combined_data_file, index=False)
print(f'complete combining: {combined_data_file}')

# Final artifact
diag_adj_df = build_diag_adj(diag_pd)
diag_adj_df.to_csv(diag_adj_file)
print(f'complete diag adjacency: {diag_adj_file}, shape={diag_adj_df.shape}')

diag_adj_df.head()